# 03. Интеграция данных из нескольких источников

## Тема

**Загрузка и интеграция данных из различных форматов. Инструменты для сбора данных. Основы Python для обработки данных**

В предыдущих ноутбуках мы научились:

- загружать CSV, Excel, JSON и HTML;
- проверять структуру таблиц;
- смотреть типы данных;
- находить пропуски, дубликаты и ошибки.

Теперь соберем несколько источников в одну аналитическую таблицу.

В этом ноутбуке разберем:

- `merge`;
- `concat`;
- `left join`;
- `inner join`;
- `outer join`;
- появление `NaN` после объединения;
- проверку результата после `merge`;
- сохранение подготовленной таблицы.

## 1. Цель ноутбука

После выполнения этого ноутбука вы должны уметь:

1. Объяснять, зачем нужно объединять таблицы.
2. Понимать, что такое ключ объединения.
3. Использовать `merge()` для соединения таблиц.
4. Отличать `left join`, `inner join` и `outer join`.
5. Понимать, почему после `merge` появляются `NaN`.
6. Проверять количество строк до и после объединения.
7. Проверять, какие ключи не нашлись в справочнике.
8. Использовать `concat()` для добавления строк.
9. Собирать итоговую таблицу `sales_prepared`.
10. Сохранять результат в CSV и Excel.

## 2. Бизнес-сюжет

Компания **«РегионМаркет»** продает товары в разных регионах.

Данные лежат в разных файлах:

```text
sales.csv              — продажи
products.xlsx          — справочник товаров
regions.json           — справочник регионов
clients.csv            — справочник клиентов
web_table_sample.html  — план продаж по регионам и каналам
```

Главная задача:

> собрать единую аналитическую таблицу, где каждая продажа будет дополнена информацией о товаре, регионе, клиенте и плане продаж.

## 3. Схема связей между таблицами

```text
sales.csv
   │
   ├── product_id → products.xlsx
   ├── region_id  → regions.json
   └── client_id  → clients.csv

sales + month + channel + region_id → web_table_sample.html
```

`sales.csv` — таблица фактов.  
Остальные таблицы — справочники или дополнительные источники.

## 4. Импорт библиотек и поиск данных

In [ ]:
import pandas as pd
from pathlib import Path

print("pandas:", pd.__version__)

In [ ]:
def find_data_dir() -> Path:
    """Найти папку с учебными данными."""
    current_dir = Path.cwd()

    candidates = [
        current_dir / "data" / "raw",
        current_dir.parent / "data" / "raw",
        current_dir.parent.parent / "data" / "raw",
    ]

    for candidate in candidates:
        if (candidate / "sales.csv").exists():
            return candidate

    return current_dir / "data" / "raw"


DATA_DIR = find_data_dir()

print("Папка с данными:")
print(DATA_DIR)

## 5. Загрузка исходных источников

Загрузим все таблицы, которые будем объединять.

In [ ]:
sales = pd.read_csv(DATA_DIR / "sales.csv")
products = pd.read_excel(DATA_DIR / "products.xlsx", sheet_name="products")
regions = pd.read_json(DATA_DIR / "regions.json")
clients = pd.read_csv(DATA_DIR / "clients.csv")
plans = pd.read_html(DATA_DIR / "web_table_sample.html")[0]

datasets = {
    "sales": sales,
    "products": products,
    "regions": regions,
    "clients": clients,
    "plans": plans,
}

for name, df in datasets.items():
    print(f"{name:<10} строк: {df.shape[0]:>3}, столбцов: {df.shape[1]:>2}")

## 6. Первичная проверка ключей

Перед объединением нужно проверить, есть ли в таблицах нужные ключевые поля.

Ключ — это столбец, по которому таблицы соединяются.

In [ ]:
for name, df in datasets.items():
    print(f"Таблица: {name}")
    print(df.columns.tolist())
    print()

Ожидаемые ключи:

| Таблица | Ключ |
|---|---|
| `sales` | `product_id`, `region_id`, `client_id` |
| `products` | `product_id` |
| `regions` | `region_id` |
| `clients` | `client_id` |
| `plans` | `region_id`, `channel`, `month` |

# Часть 1. Простая демонстрация `merge`

## 7. Что делает `merge()`

`merge()` объединяет две таблицы по общему ключу.

Простая аналогия:

> в таблице продаж есть только код товара, а в справочнике товаров есть код и название. Через `merge()` мы добавляем к продаже название товара.

In [ ]:
sales_demo = pd.DataFrame({
    "sale_id": [1, 2, 3],
    "product_id": [101, 102, 999],
    "quantity": [2, 1, 3],
})

products_demo = pd.DataFrame({
    "product_id": [101, 102, 103],
    "product_name": ["Ноутбук", "Смартфон", "Монитор"],
})

display(sales_demo)
display(products_demo)

## 8. `left join`

`left join` сохраняет все строки из левой таблицы.

В нашем случае левая таблица — продажи.  
Это значит: мы не теряем продажи, даже если товар не найден в справочнике.

In [ ]:
left_demo = sales_demo.merge(
    products_demo,
    on="product_id",
    how="left"
)

left_demo

### Что нужно заметить

Для `product_id = 999` нет строки в справочнике товаров.  
Поэтому в `product_name` появился `NaN`.

`NaN` после `merge` часто означает:

> ключ из основной таблицы не найден в справочнике.

## 9. `inner join`

`inner join` оставляет только строки, где ключ есть в обеих таблицах.

In [ ]:
inner_demo = sales_demo.merge(
    products_demo,
    on="product_id",
    how="inner"
)

inner_demo

### Что нужно заметить

Продажа с `product_id = 999` исчезла, потому что такого товара нет в справочнике.

`inner join` полезен, когда нужны только полностью сопоставленные строки.  
Но для проверки продаж его нужно использовать осторожно: можно случайно потерять часть фактов.

## 10. `outer join`

`outer join` сохраняет все ключи из обеих таблиц.

In [ ]:
outer_demo = sales_demo.merge(
    products_demo,
    on="product_id",
    how="outer",
    indicator=True
)

outer_demo

### Что показывает `_merge`

Параметр `indicator=True` добавляет столбец `_merge`.

Он показывает, откуда пришла строка:

| Значение | Что означает |
|---|---|
| `left_only` | строка есть только в левой таблице |
| `right_only` | строка есть только в правой таблице |
| `both` | строка есть в обеих таблицах |

Это удобно для диагностики проблем с ключами.

# Часть 2. Подготовка данных перед объединением

## 11. Зачем готовить данные перед `merge`

Перед объединением нужно проверить:

- ключевые столбцы существуют;
- типы ключей совпадают;
- в справочниках нет дубликатов по ключу;
- текстовые поля очищены от пробелов и регистра;
- даты приведены к формату даты;
- для планов создан общий месяц.

Если этого не сделать, после объединения могут появиться неожиданные `NaN` или лишние строки.

In [ ]:
sales_work = sales.copy()
products_work = products.copy()
regions_work = regions.copy()
clients_work = clients.copy()
plans_work = plans.copy()

print("Рабочие копии созданы.")

## 12. Вспомогательная функция для дат

В учебных данных специально есть разные форматы дат.  
Создадим функцию безопасного преобразования дат.

In [ ]:
def parse_dates_safely(series: pd.Series) -> pd.Series:
    """Преобразовать даты с учетом разных версий pandas."""
    try:
        return pd.to_datetime(series, errors="coerce", format="mixed", dayfirst=True)
    except TypeError:
        return pd.to_datetime(series, errors="coerce", dayfirst=True)

## 13. Минимальная очистка продаж

Сделаем базовую подготовку:

- дату заказа преобразуем в дату;
- создадим месяц;
- канал продаж очистим от пробелов и регистра;
- цену и количество приведем к числам;
- скидку приведем к числу и пропуски заменим на 0;
- удалим дубликаты по `sale_id`.

In [ ]:
sales_work["order_date"] = parse_dates_safely(sales_work["order_date"])
sales_work["month"] = sales_work["order_date"].dt.to_period("M").astype(str)

sales_work["channel"] = (
    sales_work["channel"]
    .astype("string")
    .str.strip()
    .str.lower()
)

sales_work["quantity"] = pd.to_numeric(sales_work["quantity"], errors="coerce")
sales_work["unit_price"] = pd.to_numeric(sales_work["unit_price"], errors="coerce")
sales_work["discount_percent"] = pd.to_numeric(sales_work["discount_percent"], errors="coerce").fillna(0)

rows_before = sales_work.shape[0]
sales_work = sales_work.drop_duplicates(subset=["sale_id"], keep="first")
rows_after = sales_work.shape[0]

print("Строк до удаления дубликатов:", rows_before)
print("Строк после удаления дубликатов:", rows_after)

sales_work.head()

## 14. Минимальная очистка справочника товаров

В справочнике товаров есть учебные проблемы:

- дубликаты `product_id`;
- ошибки в `purchase_price`;
- пробелы и разный регистр в `category`.

Для корректного `merge` справочник товаров должен иметь одну строку на один `product_id`.

In [ ]:
print("Дубликаты product_id до очистки:")
print(products_work.duplicated(subset=["product_id"]).sum())

products_work["category"] = (
    products_work["category"]
    .astype("string")
    .str.strip()
    .str.lower()
)

products_work["purchase_price"] = pd.to_numeric(products_work["purchase_price"], errors="coerce")

products_work = products_work.drop_duplicates(subset=["product_id"], keep="first")

print("\nДубликаты product_id после очистки:")
print(products_work.duplicated(subset=["product_id"]).sum())

products_work.head()

## 15. Минимальная очистка регионов и клиентов

In [ ]:
regions_work["federal_district"] = (
    regions_work["federal_district"]
    .astype("string")
    .str.strip()
    .str.lower()
)

clients_work["client_type"] = (
    clients_work["client_type"]
    .astype("string")
    .str.strip()
    .str.upper()
)

clients_work["registration_date"] = parse_dates_safely(clients_work["registration_date"])
clients_work = clients_work.drop_duplicates(subset=["client_id"], keep="first")

print("regions:", regions_work.shape)
print("clients:", clients_work.shape)

## 16. Минимальная очистка планов из HTML

План продаж находится на уровне:

```text
region_id + channel + month
```

Поэтому перед объединением нужно привести `channel` и `month` к тому же виду, что в таблице продаж.

In [ ]:
plans_work["channel"] = (
    plans_work["channel"]
    .astype("string")
    .str.strip()
    .str.lower()
)

plans_work["sales_plan"] = pd.to_numeric(plans_work["sales_plan"], errors="coerce")
plans_work["orders_plan"] = pd.to_numeric(plans_work["orders_plan"], errors="coerce")

# Приводим разные варианты месяца к формату YYYY-MM.
plans_work["month_dt"] = parse_dates_safely(plans_work["month"].astype("string"))
plans_work["month"] = plans_work["month_dt"].dt.to_period("M").astype(str)
plans_work = plans_work.drop(columns=["month_dt"])

plans_work.head()

# Часть 3. Проверка ключей до объединения

## 17. Проверка отсутствующих товаров

Проверим, какие `product_id` из продаж отсутствуют в справочнике товаров.

In [ ]:
sales_product_ids = set(sales_work["product_id"].dropna().unique())
product_reference_ids = set(products_work["product_id"].dropna().unique())

missing_product_ids = sorted(sales_product_ids - product_reference_ids)

print("product_id из продаж, которых нет в products:")
print(missing_product_ids)

## 18. Проверка отсутствующих регионов

In [ ]:
sales_region_ids = set(sales_work["region_id"].dropna().unique())
region_reference_ids = set(regions_work["region_id"].dropna().unique())

missing_region_ids = sorted(sales_region_ids - region_reference_ids)

print("region_id из продаж, которых нет в regions:")
print(missing_region_ids)

## 19. Проверка отсутствующих клиентов

In [ ]:
sales_client_ids = set(sales_work["client_id"].dropna().unique())
client_reference_ids = set(clients_work["client_id"].dropna().unique())

missing_client_ids = sorted(sales_client_ids - client_reference_ids)

print("client_id из продаж, которых нет в clients:")
print(missing_client_ids)

### Почему это важно

Если ключ есть в продажах, но отсутствует в справочнике, после `left merge` появятся `NaN`.

Это не ошибка pandas.  
Это проблема качества данных или неполного справочника.

# Часть 4. Объединение реальных источников

## 20. Продажи + товары

Объединим продажи со справочником товаров.

Используем `left join`, потому что продажи — основная таблица фактов, и мы не хотим потерять строки продаж.

In [ ]:
rows_before = sales_work.shape[0]

sales_products = sales_work.merge(
    products_work,
    on="product_id",
    how="left",
    indicator=True,
    validate="many_to_one"
)

rows_after = sales_products.shape[0]

print("Строк до merge:", rows_before)
print("Строк после merge:", rows_after)

print("\nРаспределение _merge:")
print(sales_products["_merge"].value_counts())

sales_products.head()

### Что проверяем после `merge`

1. Количество строк до и после.
2. Значения в `_merge`.
3. Пропуски в добавленных столбцах.
4. Дублирование строк.

In [ ]:
missing_products_after_merge = sales_products[sales_products["_merge"] == "left_only"]

missing_products_after_merge[["sale_id", "product_id", "product_name", "category"]]

### Почему появились `NaN`

Если у продажи есть `product_id`, которого нет в `products`, pandas не может добавить название товара, категорию и закупочную цену.

Поэтому в этих новых столбцах появляются `NaN`.

## 21. Удалим диагностический столбец `_merge`

После проверки можно удалить `_merge`, чтобы он не мешал следующему объединению.

In [ ]:
sales_products = sales_products.drop(columns=["_merge"])

sales_products.head()

## 22. Продажи + товары + регионы

Добавим справочник регионов по `region_id`.

In [ ]:
rows_before = sales_products.shape[0]

sales_products_regions = sales_products.merge(
    regions_work,
    on="region_id",
    how="left",
    indicator=True,
    validate="many_to_one"
)

rows_after = sales_products_regions.shape[0]

print("Строк до merge:", rows_before)
print("Строк после merge:", rows_after)

print("\nРаспределение _merge:")
print(sales_products_regions["_merge"].value_counts())

sales_products_regions.head()

In [ ]:
missing_regions_after_merge = sales_products_regions[sales_products_regions["_merge"] == "left_only"]

missing_regions_after_merge[["sale_id", "region_id", "region_name", "federal_district"]]

In [ ]:
sales_products_regions = sales_products_regions.drop(columns=["_merge"])

## 23. Добавляем клиентов

Теперь добавим справочник клиентов по `client_id`.

In [ ]:
rows_before = sales_products_regions.shape[0]

sales_full = sales_products_regions.merge(
    clients_work,
    on="client_id",
    how="left",
    indicator=True,
    validate="many_to_one"
)

rows_after = sales_full.shape[0]

print("Строк до merge:", rows_before)
print("Строк после merge:", rows_after)

print("\nРаспределение _merge:")
print(sales_full["_merge"].value_counts())

sales_full.head()

In [ ]:
missing_clients_after_merge = sales_full[sales_full["_merge"] == "left_only"]

missing_clients_after_merge[["sale_id", "client_id", "client_type", "loyalty_level"]]

In [ ]:
sales_full = sales_full.drop(columns=["_merge"])

# Часть 5. Объединение с планом продаж

## 24. Почему план объединяется по нескольким ключам

План продаж задан не по конкретной продаже, а по комбинации:

```text
region_id + channel + month
```

Поэтому для объединения используем список ключей.

In [ ]:
print("Ключи в sales_full:")
print(sales_full[["region_id", "channel", "month"]].head())

print("\nКлючи в plans_work:")
print(plans_work[["region_id", "channel", "month"]].head())

## 25. Факт продаж + план

Добавим к каждой продаже план по региону, каналу и месяцу.

In [ ]:
merge_keys = ["region_id", "channel", "month"]

rows_before = sales_full.shape[0]

sales_with_plan = sales_full.merge(
    plans_work,
    on=merge_keys,
    how="left",
    indicator=True,
    validate="many_to_one"
)

rows_after = sales_with_plan.shape[0]

print("Строк до merge:", rows_before)
print("Строк после merge:", rows_after)

print("\nРаспределение _merge:")
print(sales_with_plan["_merge"].value_counts())

sales_with_plan.head()

## 26. Строки, где план не найден

Если `_merge = left_only`, значит для такой комбинации региона, канала и месяца не найден план.

In [ ]:
sales_with_plan[sales_with_plan["_merge"] == "left_only"][
    ["sale_id", "region_id", "region_name", "channel", "month", "sales_plan", "orders_plan"]
].head(20)

In [ ]:
sales_with_plan = sales_with_plan.drop(columns=["_merge"])

# Часть 6. `inner join` и `outer join` на реальных данных

## 27. Сравнение `left` и `inner`

Покажем, сколько строк останется, если использовать `inner join` для товаров.

Это демонстрация: для основной сборки мы уже использовали `left join`.

In [ ]:
left_count = sales_work.merge(products_work, on="product_id", how="left").shape[0]
inner_count = sales_work.merge(products_work, on="product_id", how="inner").shape[0]

print("Строк после left join:", left_count)
print("Строк после inner join:", inner_count)
print("Потеря строк при inner join:", left_count - inner_count)

### Вывод

`inner join` может убрать продажи, если товар не найден в справочнике.

Это не всегда плохо, но такое поведение нужно осознанно контролировать.

## 28. `outer join` для диагностики ключей

`outer join` удобно использовать не как основное объединение, а как диагностический инструмент.

In [ ]:
product_key_diagnostics = sales_work[["product_id"]].drop_duplicates().merge(
    products_work[["product_id", "product_name"]],
    on="product_id",
    how="outer",
    indicator=True
)

product_key_diagnostics.sort_values(["_merge", "product_id"])

### Что показывает результат

- `both` — ключ есть и в продажах, и в справочнике;
- `left_only` — есть в продажах, но нет в справочнике;
- `right_only` — есть в справочнике, но нет в продажах.

# Часть 7. `concat`

## 29. Что делает `concat()`

`concat()` используется, когда нужно **склеить таблицы**.

Чаще всего:

- добавить строки одной таблицы под строки другой;
- объединить несколько одинаковых по структуре выгрузок;
- собрать данные за разные месяцы.

`merge()` соединяет по ключам.  
`concat()` просто склеивает таблицы.

## 30. Пример `concat`: продажи за два периода

Разделим продажи на две части и затем снова склеим.

In [ ]:
sales_january_part_1 = sales_work.iloc[:10].copy()
sales_january_part_2 = sales_work.iloc[10:20].copy()

print("Первая часть:", sales_january_part_1.shape)
print("Вторая часть:", sales_january_part_2.shape)

In [ ]:
sales_concat_example = pd.concat(
    [sales_january_part_1, sales_january_part_2],
    ignore_index=True
)

print("После concat:", sales_concat_example.shape)

sales_concat_example.head()

### Почему `ignore_index=True`

Если не указать `ignore_index=True`, pandas сохранит старые индексы строк.

Часто после склейки нескольких выгрузок удобнее создать новый последовательный индекс.

## 31. `concat` с признаком источника

Иногда полезно знать, из какой части пришла строка.

In [ ]:
sales_january_part_1["source_file"] = "sales_part_1.csv"
sales_january_part_2["source_file"] = "sales_part_2.csv"

sales_concat_with_source = pd.concat(
    [sales_january_part_1, sales_january_part_2],
    ignore_index=True
)

sales_concat_with_source[["sale_id", "order_date", "source_file"]].head(15)

# Часть 8. Расчет итоговых показателей

## 32. Создаем расчетные поля

После объединения можно рассчитать:

- выручку до скидки;
- сумму скидки;
- выручку после скидки;
- себестоимость;
- валовую прибыль;
- выполнение плана.

In [ ]:
sales_prepared = sales_with_plan.copy()

sales_prepared["gross_revenue"] = sales_prepared["quantity"] * sales_prepared["unit_price"]
sales_prepared["discount_amount"] = sales_prepared["gross_revenue"] * sales_prepared["discount_percent"] / 100
sales_prepared["net_revenue"] = sales_prepared["gross_revenue"] - sales_prepared["discount_amount"]
sales_prepared["purchase_cost"] = sales_prepared["quantity"] * sales_prepared["purchase_price"]
sales_prepared["gross_profit"] = sales_prepared["net_revenue"] - sales_prepared["purchase_cost"]
sales_prepared["plan_completion_rate"] = sales_prepared["net_revenue"] / sales_prepared["sales_plan"]

sales_prepared[[
    "sale_id",
    "quantity",
    "unit_price",
    "discount_percent",
    "gross_revenue",
    "net_revenue",
    "purchase_cost",
    "gross_profit",
    "sales_plan",
    "plan_completion_rate"
]].head(10)

## 33. Финальная проверка подготовленной таблицы

Перед сохранением результата проверим:

- размер;
- список столбцов;
- пропуски;
- дубликаты `sale_id`.

In [ ]:
print("Размер sales_prepared:", sales_prepared.shape)

print("\nКоличество дубликатов sale_id:")
print(sales_prepared.duplicated(subset=["sale_id"]).sum())

print("\nПропуски по столбцам:")
sales_prepared.isna().sum()

## 34. Проверка ключевых аналитических полей

Посмотрим, где не удалось добавить справочную информацию или план.

In [ ]:
key_quality_checks = {
    "Без названия товара": sales_prepared["product_name"].isna().sum(),
    "Без региона": sales_prepared["region_name"].isna().sum(),
    "Без типа клиента": sales_prepared["client_type"].isna().sum(),
    "Без плана продаж": sales_prepared["sales_plan"].isna().sum(),
    "Без net_revenue": sales_prepared["net_revenue"].isna().sum(),
    "Без gross_profit": sales_prepared["gross_profit"].isna().sum(),
}

for check_name, value in key_quality_checks.items():
    print(f"{check_name:<25}: {value}")

### Что означает эта проверка

Если есть пропуски в `product_name`, `region_name`, `client_type` или `sales_plan`, значит не все данные удалось сопоставить со справочниками.

Это не всегда повод удалять строки.  
Но это обязательно нужно указать в аналитическом выводе.

# Часть 9. Сохранение подготовленной таблицы

## 35. Сохраняем `sales_prepared.csv`

Итоговая таблица пригодится в следующих ноутбуках:

- базовый анализ;
- визуализация;
- мост Python → R.

In [ ]:
output_dir = Path("data/prepared")
output_dir.mkdir(parents=True, exist_ok=True)

prepared_csv_path = output_dir / "sales_prepared.csv"

sales_prepared.to_csv(prepared_csv_path, index=False, encoding="utf-8")

print("Файл сохранен:")
print(prepared_csv_path)

print("\nПроверка:")
print(prepared_csv_path.exists())

## 36. Дополнительно сохраняем Excel-версию

Excel-версия удобна для просмотра результата вручную.

In [ ]:
output_excel_dir = Path("data/output")
output_excel_dir.mkdir(parents=True, exist_ok=True)

prepared_excel_path = output_excel_dir / "sales_prepared.xlsx"

sales_prepared.to_excel(prepared_excel_path, index=False, sheet_name="sales_prepared")

print("Файл сохранен:")
print(prepared_excel_path)

print("\nПроверка:")
print(prepared_excel_path.exists())

# Часть 10. Мини-задания

Выполните задания самостоятельно.

## Задание 1

Объедините `sales_work` и `products_work` через `inner join`.

Сравните количество строк с `left join`.

In [ ]:
# Ваш код здесь

## Задание 2

Найдите строки продаж, у которых не найден товар в справочнике `products`.

In [ ]:
# Ваш код здесь

## Задание 3

Объедините `sales_work` и `clients_work` по `client_id`.

Используйте `how="left"` и `indicator=True`.

In [ ]:
# Ваш код здесь

## Задание 4

Найдите клиентов из продаж, которых нет в справочнике клиентов.

In [ ]:
# Ваш код здесь

## Задание 5

С помощью `concat()` склейте первые 5 строк и следующие 5 строк таблицы `sales_work`.

Проверьте размер результата.

In [ ]:
# Ваш код здесь

## Задание 6

Сохраните первые 20 строк `sales_prepared` в файл:

```text
data/output/sales_prepared_sample.csv
```

In [ ]:
# Ваш код здесь

# Часть 11. Контрольные вопросы

Ответьте своими словами:

1. Что делает `merge()`?
2. Что такое ключ объединения?
3. Чем `left join` отличается от `inner join`?
4. Что делает `outer join`?
5. Почему после `merge` появляются `NaN`?
6. Зачем проверять количество строк до и после объединения?
7. Зачем нужен `indicator=True`?
8. Что показывает столбец `_merge`?
9. Чем `concat()` отличается от `merge()`?
10. Почему справочники нужно проверять на дубликаты по ключу?
11. Почему для основной таблицы продаж чаще используют `left join`?
12. Какие проверки нужно сделать перед сохранением подготовленной таблицы?

# 37. Итог ноутбука

В этом ноутбуке мы научились:

- объединять таблицы через `merge`;
- использовать `left join`, `inner join`, `outer join`;
- понимать появление `NaN` после объединения;
- проверять ключи до и после `merge`;
- использовать `indicator=True` для диагностики;
- использовать `validate="many_to_one"` для контроля типа связи;
- использовать `concat()` для склейки строк;
- собирать единую таблицу `sales_prepared`;
- сохранять подготовленную таблицу в CSV и Excel.

Следующий шаг:

```text
04_basic_analysis.ipynb
```

В нем мы будем анализировать уже подготовленную таблицу: считать выручку, прибыль, группировки и сводные таблицы.